# Standard ML vs. Deep Learning Model Comparison

This notebook evaluates the saved Random Forest baseline and saved neural-network model using their existing test-data setups. The baseline uses the preprocessed test CSVs; the deep-learning model uses a stratified 20% split from the raw data. Because the rows and preprocessing setups differ, this is not a same-row head-to-head comparison.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

from deep_learning.src.data_processing import DataProcessingConfig

ROOT = Path.cwd()
if not (ROOT / "models/baseline/best_model.joblib").exists():
    ROOT = ROOT.parent

assert (ROOT / "models/baseline/best_model.joblib").exists(), "Run this notebook from the repository or its notebooks directory."
METRIC_NAMES = ("Accuracy", "Precision", "Recall", "F1", "ROC AUC")

In [2]:
def calculate_metrics(y_true, y_pred, y_probability):
    """Calculate classification metrics from labels and positive-class scores."""
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_true, y_probability),
    }

In [3]:
def evaluate_standard_ml():
    """Evaluate the saved baseline on its preprocessed test set."""
    model = joblib.load(ROOT / "models/baseline/best_model.joblib")
    x_test = pd.read_csv(ROOT / "data/preprocessed/x_test.csv")
    y_test = pd.read_csv(ROOT / "data/preprocessed/y_test.csv").squeeze().astype(int)

    predictions = model.predict(x_test)
    probabilities = model.predict_proba(x_test)[:, 1]
    return calculate_metrics(y_test.to_numpy(), predictions, probabilities)


standard_ml_metrics = evaluate_standard_ml()
standard_ml_metrics

{'Accuracy': 0.6609,
 'Precision': 0.612053659498915,
 'Recall': 0.6853324497459686,
 'F1': 0.6466235931638182,
 'ROC AUC': 0.7290151431585488}

In [4]:
def evaluate_deep_learning():
    """Evaluate the saved network on its stratified raw-data test split."""
    raw = pd.read_csv(ROOT / "data/raw/order_return_data.csv")
    x_raw = raw.drop(columns="return")
    y = raw["return"].astype(int)
    _, x_test_raw, _, y_test = train_test_split(
        x_raw, y, test_size=0.2, random_state=42, stratify=y
    )

    processor = DataProcessingConfig(load_data=False)
    processor.load(ROOT / "deep_learning/models/preprocessor.joblib")
    x_test = processor.transform(x_test_raw)

    state_dict = torch.load(
        ROOT / "deep_learning/models/best_model.pth",
        map_location="cpu",
        weights_only=True,
    )
    model = nn.Sequential(
        nn.Linear(state_dict["0.weight"].shape[1], 128),
        nn.ReLU(),
        nn.Linear(128, 1),
    )
    model.load_state_dict(state_dict)
    model.eval()

    with torch.no_grad():
        probabilities = torch.sigmoid(model(x_test)).numpy().ravel()
    predictions = (probabilities > 0.5).astype(int)

    return calculate_metrics(y_test.to_numpy(), predictions, probabilities)


deep_learning_metrics = evaluate_deep_learning()
deep_learning_metrics

{'Accuracy': 0.64045,
 'Precision': 0.5939076630176107,
 'Recall': 0.6808162374508948,
 'F1': 0.634399308556612,
 'ROC AUC': 0.7018130231462483}

In [5]:
results = {
    "Standard ML": standard_ml_metrics,
    "Deep learning": deep_learning_metrics,
}

comparison = pd.DataFrame(results).loc[list(METRIC_NAMES)]
display(comparison.style.format("{:.4f}").set_caption("Saved model test metrics"))

print(
    "Note: the models use different saved test-set/preprocessing setups; "
    "this is not a same-row comparison."
)

,Standard ML,Deep learning
Accuracy,0.6609,0.6404
Precision,0.6121,0.5939
Recall,0.6853,0.6808
F1,0.6466,0.6344
ROC AUC,0.7290,0.7018


Note: the models use different saved test-set/preprocessing setups; this is not a same-row comparison.
